# GPflow syntax: VGP
This is the model which the VWP model heavily relies on.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

In [ ]:
import matplotlib
import numpy as np
import tensorflow as tf

In [ ]:
import gpflow
from gpflow.utilities import print_summary

In [ ]:
sys.path.append('../../FCEst-benchmarking')

from helpers.synthetic_covariance_structures import get_constant_covariances, get_periodic_covariances, get_stepwise_covariances
from helpers.synthetic_covariance_structures import get_d2_covariance_structure

In [ ]:
%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (12, 6)
plt = matplotlib.pyplot

## Generate bivariate data

In [ ]:
N = 400

In [ ]:
cov_structure = get_d2_covariance_structure(
    get_periodic_covariances(n_samples=N, n_periods=1)
)
cov_structure.shape

In [ ]:
x = np.linspace(0, 1, N).reshape(-1, 1)
x.shape

In [ ]:
ts = cov_structure[:, 0, 1].reshape(-1, 1)
ts.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'x-')
plt.xlabel('time [steps]')

# Variational Gaussian Process Regression
This is an example to see the syntax of GPflow.

In [ ]:
k = gpflow.kernels.Matern52()
k

In [ ]:
print_summary(k)

In [ ]:
m = gpflow.models.VGP(
    data=(x, ts),
    kernel=k,
    likelihood=gpflow.likelihoods.Gaussian(),  # unlike GPR, we now have to specify a likelihood as well
    mean_function=None
)

In [ ]:
m.likelihood.variance.assign(0.01)
m.kernel.lengthscales.assign(0.3)
m

In [ ]:
# predict mean and variance of latent GP at test points
mean, var = m.predict_f(x)
mean.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'kx')
plt.plot(x, mean, "C0", lw=2)
plt.fill_between(
    x[:, 0],
    mean[:, 0] - 1.96 * np.sqrt(var[:, 0]),
    mean[:, 0] + 1.96 * np.sqrt(var[:, 0]),
    color="C0",
    alpha=0.2,
)
plt.xlabel('time [steps]')

## Optimization
We do not use minibatches here (if we want to do that we may need to alter the code or move to SVGP). Note that in the case of VGP our data is part of the model class (in contrast with SVGP).

In [ ]:
m.log_prior_density()

In [ ]:
m.log_posterior_density()

In [ ]:
m.elbo()

In [ ]:
# m.trainable_variables

In [ ]:
n_iterations = 5000
log_interval = 10

In [ ]:
def run_adam(model, iterations):
    """
    Utility function running the Adam optimizer
    :param model: GPflow model
    :param iterations: number of iterations
    """
    # Create an Adam Optimizer action
    logf = []
    training_loss = model.training_loss_closure(compile=True)
    optimizer = tf.optimizers.Adam()

    @tf.function
    def optimization_step():
        optimizer.minimize(training_loss, model.trainable_variables)

    for step in range(iterations):
        optimization_step()
        if step % log_interval == 0:
            elbo = -training_loss().numpy()
            logf.append(elbo)
    return logf

In [ ]:
# %debug

In [ ]:
%%time

maxiter = ci_niter(n_iterations)
logf = run_adam(m, maxiter)

In [ ]:
plt.plot(np.arange(maxiter)[::log_interval], logf)
plt.xlabel("iteration")
_ = plt.ylabel("ELBO")

In [ ]:
m

In [ ]:
# predict mean and variance of latent GP at test points
mean, var = m.predict_f(x)
mean.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'kx')
plt.plot(x, mean, "C0", lw=2)
plt.fill_between(
    x[:, 0],
    mean[:, 0] - 1.96 * np.sqrt(var[:, 0]),
    mean[:, 0] + 1.96 * np.sqrt(var[:, 0]),
    color="C0",
    alpha=0.2,
)
plt.xlabel('time [steps]')

In [ ]:
m.log_prior_density()

In [ ]:
m.log_posterior_density()

In [ ]:
m.elbo()